In [26]:
import pandas as pd
import sqlite3
from datetime import date

In [62]:
food_listings = pd.read_csv("food_listings_data.csv")
providers = pd.read_csv("providers_data.csv")
receivers = pd.read_csv("receivers_data.csv")
claims = pd.read_csv("claims_data.csv")

In [63]:
food_listings[food_listings.duplicated()]

,Food_ID,Food_Name,Quantity,Expiry_Date,Provider_ID,Provider_Type,Location,Food_Type,Meal_Type


In [64]:
providers[providers.duplicated()]

,Provider_ID,Name,Type,Address,City,Contact


In [65]:
receivers[receivers.duplicated()]

,Receiver_ID,Name,Type,City,Contact


In [66]:
claims[receivers.duplicated()]

,Claim_ID,Food_ID,Receiver_ID,Status,Timestamp


In [67]:
food_listings.columns

Index(['Food_ID', 'Food_Name', 'Quantity', 'Expiry_Date', 'Provider_ID',
       'Provider_Type', 'Location', 'Food_Type', 'Meal_Type'],
      dtype='object')

In [68]:
providers.columns

Index(['Provider_ID', 'Name', 'Type', 'Address', 'City', 'Contact'], dtype='object')

In [69]:
receivers.columns

Index(['Receiver_ID', 'Name', 'Type', 'City', 'Contact'], dtype='object')

In [70]:
claims.columns

Index(['Claim_ID', 'Food_ID', 'Receiver_ID', 'Status', 'Timestamp'], dtype='object')

In [77]:
food_listings['Food_ID'].is_unique

True

In [78]:
providers['Provider_ID'].is_unique

True

In [79]:
receivers['Receiver_ID'].is_unique

True

In [80]:
claims['Claim_ID'].is_unique

True

In [81]:
import sqlite3
conn = sqlite3.connect("food_redistribution.db")
cursor = conn.cursor()

In [82]:
cursor.executescript("""
DROP TABLE IF EXISTS Providers;
DROP TABLE IF EXISTS Receivers;
DROP TABLE IF EXISTS Food_Listings;
DROP TABLE IF EXISTS Claims;
""")

In [83]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS food_listings (
    listing_id INTEGER PRIMARY KEY AUTOINCREMENT,
    provider_id INTEGER,
    food_type TEXT,
    meal_type TEXT,
    quantity INTEGER,
    expiry_date TEXT,
    city TEXT,
    claim_status TEXT
)
""")

conn.commit()
print("Tables created successfully.")

Tables created successfully.


In [84]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS providers (
    Provider_ID INTEGER PRIMARY KEY,
    Name TEXT,
    Type TEXT,
    Address TEXT,
    City TEXT,
    Contact TEXT
)
""")
conn.commit()
print("Tables created successfully.")

Tables created successfully.


In [85]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS receivers (
    Receiver_ID INTEGER PRIMARY KEY,
    Name TEXT,
    Type TEXT,
    City TEXT,
    Contact TEXT
)
""")
conn.commit()
print("Tables created successfully.")

Tables created successfully.


In [86]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS claims (
    Claim_ID INTEGER PRIMARY KEY,
    Food_ID INTEGER,
    Receiver_ID INTEGER,
    Status TEXT,
    Timestamp TEXT
);
""")
conn.commit()
print("Tables created successfully.")

Tables created successfully.


In [87]:
providers.to_sql("providers", conn, if_exists="replace", index=False)
receivers.to_sql("receivers", conn, if_exists="replace", index=False)
food_listings.to_sql("food_listings", conn, if_exists="replace", index=False)
claims.to_sql("claims", conn, if_exists="replace", index=False)

1000

In [88]:
providers.to_sql("providers", conn, if_exists="append", index=False)
receivers.to_sql("receivers", conn, if_exists="append", index=False)
food_listings.to_sql("food_listings", conn, if_exists="append", index=False)
claims.to_sql("claims", conn, if_exists="append", index=False)

print("CSV data inserted successfully.")

CSV data inserted successfully.


In [89]:
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

tables


,name
0,sqlite_sequence
1,providers
2,receivers
3,food_listings
4,claims


In [90]:
#Q1 ---> How many food providers and receivers are there in each city?

pd.read_sql("""
SELECT City, COUNT(*) AS count, 'Providers' AS type
FROM Providers
GROUP BY City

UNION ALL

SELECT City, COUNT(*) AS count, 'Receivers' AS type
FROM Receivers
GROUP BY City;
""", conn)


,City,count,type
0,Adambury,2,Providers
1,Adamsview,2,Providers
2,Adamsville,2,Providers
3,Aguirreville,2,Providers
4,Alexanderchester,2,Providers
...,...,...,...
1924,Woodardview,2,Receivers
1925,Woodsfurt,2,Receivers
1926,Wrightland,2,Receivers
1927,Wyattton,2,Receivers


In [91]:
pd.read_sql("""
SELECT City, COUNT(*) AS receiver_count
FROM Receivers
GROUP BY City;
""", conn)

,City,receiver_count
0,Aaronshire,2
1,Adamland,2
2,Aguilarbury,2
3,Aguilarstad,2
4,Alexanderbury,2
...,...,...
961,Woodardview,2
962,Woodsfurt,2
963,Wrightland,2
964,Wyattton,2


In [92]:
#Q2---> Which type of food provider (restaurant, grocery store, etc.) contributes the most food?

pd.read_sql("""
SELECT Provider_Type, SUM(Quantity)
FROM food_listings
GROUP BY Provider_Type;
""", conn)


,Provider_Type,SUM(Quantity)
0,Catering Service,12232
1,Grocery Store,12118
2,Restaurant,13846
3,Supermarket,13392


In [93]:
#Q3 ---> What is the contact information of food providers in a specific city?

pd.read_sql("""
SELECT Name, Contact
FROM providers
WHERE City = 'New Jessica';
""", conn)

,Name,Contact
0,Gonzales-Cochran,+1-600-220-0480
1,Gonzales-Cochran,+1-600-220-0480


In [94]:
#Q4 ---> Which receivers have claimed the most food?

pd.read_sql("""
SELECT Receiver_ID, COUNT(*) AS Total_Claims
FROM Claims
GROUP BY Receiver_ID
ORDER BY Total_Claims DESC;
""", conn)


,Receiver_ID,Total_Claims
0,800,10
1,742,10
2,371,10
3,276,10
4,901,8
...,...,...
619,15,2
620,12,2
621,7,2
622,3,2


In [95]:
#Q5---> What is the total quantity of food available from all providers?

pd.read_sql("""
SELECT SUM(Quantity) AS total_food_available
FROM food_listings;
""", conn)


,total_food_available
0,51588


In [96]:
#Q6---> Which city has the highest number of food listings?

pd.read_sql("""
SELECT Location, COUNT(*) AS total_listings
FROM food_listings
GROUP BY Location
ORDER BY total_listings DESC;
""", conn)


,Location,total_listings
0,South Kathryn,12
1,New Carol,12
2,Perezport,10
3,Jimmyberg,10
4,East Angela,10
...,...,...
619,Andersonmouth,2
620,Amandashire,2
621,Allenborough,2
622,Alexanderchester,2


In [97]:
pd.read_sql("""
SELECT Food_Type, COUNT(*) AS count
FROM food_listings
GROUP BY Food_Type
ORDER BY count DESC;
""", conn)


,Food_Type,count
0,Vegetarian,672
1,Vegan,668
2,Non-Vegetarian,660


In [98]:
# Q8 ---> Number of claims per food item

pd.read_sql("""
SELECT f.Food_Name, COUNT(c.Claim_ID) AS total_claims
FROM claims c
JOIN food_listings f
ON c.Food_ID = f.Food_ID
GROUP BY f.Food_Name;
""", conn)


,Food_Name,total_claims
0,Bread,376
1,Chicken,408
2,Dairy,440
3,Fish,432
4,Fruits,284
5,Pasta,348
6,Rice,488
7,Salad,424
8,Soup,456
9,Vegetables,344


In [99]:
#Q9 Which provider has had the highest number of successful food claims?

pd.read_sql("""
SELECT providers.Name, COUNT(*) AS claims_count
FROM claims
JOIN food_listings
ON claims.Food_ID = food_listings.Food_ID
JOIN providers
ON food_listings.Provider_ID = providers.Provider_ID
GROUP BY providers.Name
ORDER BY claims_count DESC
LIMIT 1;
""", conn)


,Name,claims_count
0,Butler-Richardson,96


In [100]:
#Q10. What percentage of food claims are completed vs. pending vs. canceled?

pd.read_sql("""
SELECT Status,
       COUNT(*) * 100.0 / (SELECT COUNT(*) FROM claims) AS percentage
FROM claims
GROUP BY Status;
""", conn)


,Status,percentage
0,Cancelled,33.6
1,Completed,33.9
2,Pending,32.5


In [101]:
#Q11. What is the average quantity of food claimed per receiver?

pd.read_sql("""
SELECT r.Name, AVG(f.Quantity) AS avg_quantity_claimed
FROM claims c
JOIN food_listings f
ON c.Food_ID = f.Food_ID
JOIN receivers r
ON c.Receiver_ID = r.Receiver_ID
GROUP BY r.Name;
""", conn)


,Name,avg_quantity_claimed
0,Aaron Keller,39.000000
1,Aaron Rios,21.000000
2,Aaron Scott,45.000000
3,Abigail Crawford,25.666667
4,Adam Browning,5.000000
...,...,...
615,William Barnes,47.000000
616,William Fox,12.000000
617,William Frederick,21.400000
618,Yvette Huffman,45.000000


In [102]:
#Q12. Which meal type (breakfast, lunch, dinner, snacks) is claimed the most?

pd.read_sql("""
SELECT f.Meal_Type, COUNT(*) AS claims_count
FROM claims c
JOIN food_listings f
ON c.Food_ID = f.Food_ID
GROUP BY f.Meal_Type
ORDER BY claims_count DESC;
""", conn)


,Meal_Type,claims_count
0,Breakfast,1112
1,Lunch,1000
2,Snacks,960
3,Dinner,928


In [103]:
#Q13 What is the total quantity of food donated by each provider?

pd.read_sql("""
SELECT p.Name, SUM(f.Quantity) AS total_food_donated
FROM food_listings f
JOIN providers p
ON f.Provider_ID = p.Provider_ID
GROUP BY p.Name;
""", conn)


,Name,total_food_donated
0,"Abbott, Brooks and Moreno",188
1,Abbott-Robinson,176
2,Adams-Meyer,112
3,Adams-Young,64
4,Aguilar Group,264
...,...,...
623,"Young, Townsend and Mccarthy",348
624,Young-Luna,140
625,Young-Wu,8
626,Yu-Rodriguez,56


In [104]:
#Q14 Which type of receiver (NGO, Community Center, Individual) prefers which food types?

pd.read_sql("""
SELECT r.Type AS receiver_type, f.Food_Type, COUNT(*) AS claims_count
FROM claims c
JOIN food_listings f
ON c.Food_ID = f.Food_ID
JOIN receivers r
ON c.Receiver_ID = r.Receiver_ID
GROUP BY r.Type, f.Food_Type
ORDER BY claims_count DESC;
""", conn)


,receiver_type,Food_Type,claims_count
0,NGO,Vegetarian,864
1,Charity,Non-Vegetarian,728
2,Charity,Vegetarian,720
3,Charity,Vegan,696
4,Individual,Non-Vegetarian,680
5,NGO,Non-Vegetarian,672
6,Shelter,Vegan,648
7,NGO,Vegan,640
8,Shelter,Vegetarian,624
9,Individual,Vegetarian,592


In [105]:
#Q15 What is the total quantity of food donated by each provider

pd.read_sql("""
SELECT p.Name, AVG(f.Quantity) AS avg_food_provided
FROM food_listings f
JOIN providers p
ON f.Provider_ID = p.Provider_ID
GROUP BY p.Name;
""", conn)


,Name,avg_food_provided
0,"Abbott, Brooks and Moreno",47.000000
1,Abbott-Robinson,44.000000
2,Adams-Meyer,28.000000
3,Adams-Young,16.000000
4,Aguilar Group,33.000000
...,...,...
623,"Young, Townsend and Mccarthy",29.000000
624,Young-Luna,35.000000
625,Young-Wu,2.000000
626,Yu-Rodriguez,14.000000


In [106]:
pip install streamlit-jupyter

  Using cached streamlit_jupyter-0.3.1-py3-none-any.whl.metadata (6.4 kB)
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached jupyter-1.1.1-py2.py3-none-any.whl.metadata (2.0 kB)
  Using cached stqdm-0.0.5-py3-none-any.whl.metadata (3.0 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
  Using cached jupyter_console-6.6.3-py3-none-any.whl.metadata (5.8 kB)
  Using cached nbconvert-7.17.0-py3-none-any.whl.metadata (8.4 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jupyter_lsp-2.3.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached jupyter_server-2.17.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached jupyterlab_server-2.28.0-py3-none-any.whl.metadata (5.9 kB)
  Using cached notebook_shim-0.2.4-py3-none-any.whl.metadata (4.0 kB)
  Using cached beautifulsoup4-


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [108]:
import streamlit as st
from streamlit_jupyter import StreamlitPatcher

In [ ]:
%%writefile FeedForward.py
import streamlit as st
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# -----------------------------
# Page Config
# -----------------------------
st.set_page_config(page_title="Food Redistribution Dashboard", layout="wide")

# -----------------------------
# Load CSV and Remove Duplicates
# -----------------------------
@st.cache_data
def load_clean_data():
    food = pd.read_csv("food_listings_data.csv").drop_duplicates(subset="Food_ID")
    providers = pd.read_csv("providers_data.csv").drop_duplicates(subset="Provider_ID")
    receivers = pd.read_csv("receivers_data.csv").drop_duplicates(subset="Receiver_ID")
    claims = pd.read_csv("claims_data.csv").drop_duplicates(subset="Claim_ID")
    return food, providers, receivers, claims

food_df, providers_df, receivers_df, claims_df = load_clean_data()

# -----------------------------
# Save Clean Data to SQLite
# -----------------------------
conn = sqlite3.connect("food_redistribution.db")

food_df.to_sql("food_listings", conn, if_exists="replace", index=False)
providers_df.to_sql("providers", conn, if_exists="replace", index=False)
receivers_df.to_sql("receivers", conn, if_exists="replace", index=False)
claims_df.to_sql("claims", conn, if_exists="replace", index=False)

# -----------------------------
# Header
# -----------------------------
col1, col2 = st.columns([6,1])

with col1:
    st.markdown("""
        <h1 style='margin-bottom:0;'>Food Redistribution Dashboard</h1>
        <p style='margin-top:0; font-size:18px; color:gray;'>
        Precision Logistics for a Zero-Waste World
        </p>
    """, unsafe_allow_html=True)

with col2:
    st.image("D:\Downloads\Gemini_Generated_Image_hqirgmhqirgmhqir.png", width=80)


st.markdown("---")

# -----------------------------
# Load Main Tables
# -----------------------------
food_df = pd.read_sql("SELECT * FROM food_listings", conn)
claims_df = pd.read_sql("SELECT * FROM claims", conn)
providers_df = pd.read_sql("SELECT * FROM providers", conn)

# -----------------------------
# KPI Cards (Gradient Style)
# -----------------------------
st.subheader("Key Metrics")

k1, k2, k3 = st.columns(3)

k1.markdown(f"""
<div style="background: linear-gradient(135deg,#43cea2,#185a9d);
padding:20px;border-radius:12px;color:white;text-align:center">
<h4>📦 Total Listings</h4>
<h2>{len(food_df)}</h2>
</div>
""", unsafe_allow_html=True)

k2.markdown(f"""
<div style="background: linear-gradient(135deg,#36d1dc,#5b86e5);
padding:20px;border-radius:12px;color:white;text-align:center">
<h4>📑 Total Claims</h4>
<h2>{len(claims_df)}</h2>
</div>
""", unsafe_allow_html=True)

k3.markdown(f"""
<div style="background: linear-gradient(135deg,#ff9966,#ff5e62);
padding:20px;border-radius:12px;color:white;text-align:center">
<h4>🏢 Active Providers</h4>
<h2>{providers_df['Provider_ID'].nunique()}</h2>
</div>
""", unsafe_allow_html=True)

st.markdown("---")

# -----------------------------
# Dataset Selection
# -----------------------------
table_options = {
    "Food Catalogue": "food_listings",
    "Claims Data": "claims",
    "Providers Data": "providers",
    "Receivers Data": "receivers"
}

selected_view = st.selectbox("Select Dataset", list(table_options.keys()))
selected_table = table_options[selected_view]

df = pd.read_sql(f"SELECT * FROM {selected_table}", conn)

# -----------------------------
# Tabs Layout
# -----------------------------
tab1, tab2, = st.tabs(["Table View"," ANALYTICS - SQL Queries"])

# =============================
# TAB 1 – TABLE VIEW
# =============================
with tab1:
    st.subheader(selected_view)

    filtered_df = df.copy()

    for col in df.columns:
        unique_vals = df[col].dropna().unique()
        if len(unique_vals) < 20:
            selected_val = st.selectbox(f"Filter by {col}", ["All"] + list(unique_vals), key=col)
            if selected_val != "All":
                filtered_df = filtered_df[filtered_df[col] == selected_val]

    st.dataframe(filtered_df, use_container_width=True)


# =============================
# TAB 2 – ANALYTICS
# =============================
with tab2:

    st.subheader("SQL Insights")

    query_option = st.selectbox(
        "Choose Analysis",
        (
            "Total Food Available",
            "Providers per City",
            "Most Common Food Type",
            "Provider with Most Claims"
        )
    )

    if query_option == "Total Food Available":
        result = pd.read_sql("SELECT SUM(Quantity) AS Total_Food FROM food_listings", conn)
        st.dataframe(result)
        st.bar_chart(result.set_index(result.columns[0]))

    elif query_option == "Providers per City":
        result = pd.read_sql("""
            SELECT City, COUNT(*) AS Provider_Count
            FROM providers
            GROUP BY City
        """, conn)
        st.dataframe(result)
        st.bar_chart(result.set_index("City"))

    elif query_option == "Most Common Food Type":
        result = pd.read_sql("""
            SELECT Food_Type, COUNT(*) AS Count
            FROM food_listings
            GROUP BY Food_Type
            ORDER BY Count DESC
        """, conn)
        st.dataframe(result)
        st.bar_chart(result.set_index("Food_Type"))

    elif query_option == "Provider with Most Claims":
        result = pd.read_sql("""
            SELECT p.Name, COUNT(c.Claim_ID) AS claim_count
            FROM claims c
            JOIN food_listings f ON c.Food_ID = f.Food_ID
            JOIN providers p ON f.Provider_ID = p.Provider_ID
            GROUP BY p.Name
            ORDER BY claim_count DESC
            LIMIT 1
        """, conn)
        st.dataframe(result)

# -----------------------------
# Expiry Alerts
# -----------------------------
if "Expiry_Date" in food_df.columns:
    food_df["Expiry_Date"] = pd.to_datetime(food_df["Expiry_Date"], errors="coerce")
    soon_expiring = food_df[
        food_df["Expiry_Date"] <= datetime.now() + timedelta(days=2)
    ]

    if not soon_expiring.empty:
        st.error("⚠️ Food items expiring within the next 48 hours!")
        st.dataframe(soon_expiring, use_container_width=True)

st.markdown("---")

st.subheader("SQL Query Results")

query = st.selectbox(
    "Choose a question",
    (
        "Q1: Total Food Available",
        "Q2: Providers per City",
        "Q3: Receivers per City",
        "Q4: Provider-wise Total Food Donated",
        "Q5: Average Food Provided by Each Provider",
        "Q6: City with Highest Food Listings",
        "Q7: Most Common Food Type",
        "Q8: Claims per Food Item",
        "Q9: Provider with Most Claims",
        "Q10: Average Food Claimed per Receiver",
        "Q11: Most Claimed Meal Type",
        "Q12: City with Highest Claims",
        "Q13: Food Type Claimed the Most",
        "Q14: Average Quantity per Listing",
        "Q15: Provider-wise Average Donation"
    ),
    key="final_sql_section"
)

# Helper function to show chart automatically
def show_chart(df):
    if df.shape[1] >= 2:
        try:
            chart_df = df.set_index(df.columns[0])
            st.bar_chart(chart_df)
        except:
            pass


if query == "Q1: Total Food Available":
    df = pd.read_sql("""
        SELECT SUM(Quantity) AS total_food_available
        FROM food_listings
    """, conn)
    st.dataframe(df)
    show_chart(df)


elif query == "Q2: Providers per City":
    df = pd.read_sql("""
        SELECT City, COUNT(*) AS provider_count
        FROM providers
        GROUP BY City
    """, conn)
    st.dataframe(df)
    show_chart(df)


elif query == "Q3: Receivers per City":
    df = pd.read_sql("""
        SELECT City, COUNT(*) AS receiver_count
        FROM receivers
        GROUP BY City
    """, conn)
    st.dataframe(df)
    show_chart(df)


elif query == "Q4: Provider-wise Total Food Donated":
    df = pd.read_sql("""
        SELECT p.Name, SUM(f.Quantity) AS total_food
        FROM food_listings f
        JOIN providers p ON f.Provider_ID = p.Provider_ID
        GROUP BY p.Name
    """, conn)
    st.dataframe(df)
    show_chart(df)


elif query == "Q5: Average Food Provided by Each Provider":
    df = pd.read_sql("""
        SELECT p.Name, AVG(f.Quantity) AS avg_food
        FROM food_listings f
        JOIN providers p ON f.Provider_ID = p.Provider_ID
        GROUP BY p.Name
    """, conn)
    st.dataframe(df)
    show_chart(df)


elif query == "Q6: City with Highest Food Listings":
    df = pd.read_sql("""
        SELECT Location, COUNT(*) AS listing_count
        FROM food_listings
        GROUP BY Location
        ORDER BY listing_count DESC
        LIMIT 1
    """, conn)
    st.dataframe(df)


elif query == "Q7: Most Common Food Type":
    df = pd.read_sql("""
        SELECT Food_Type, COUNT(*) AS count
        FROM food_listings
        GROUP BY Food_Type
        ORDER BY count DESC
    """, conn)
    st.dataframe(df)
    show_chart(df)


elif query == "Q8: Claims per Food Item":
    df = pd.read_sql("""
        SELECT f.Food_Name, COUNT(c.Claim_ID) AS total_claims
        FROM claims c
        JOIN food_listings f ON c.Food_ID = f.Food_ID
        GROUP BY f.Food_Name
    """, conn)
    st.dataframe(df)
    show_chart(df)


elif query == "Q9: Provider with Most Claims":
    df = pd.read_sql("""
        SELECT p.Name, COUNT(c.Claim_ID) AS claim_count
        FROM claims c
        JOIN food_listings f ON c.Food_ID = f.Food_ID
        JOIN providers p ON f.Provider_ID = p.Provider_ID
        GROUP BY p.Name
        ORDER BY claim_count DESC
        LIMIT 1
    """, conn)
    st.dataframe(df)


elif query == "Q10: Average Food Claimed per Receiver":
    df = pd.read_sql("""
        SELECT r.Name, AVG(f.Quantity) AS avg_quantity
        FROM claims c
        JOIN food_listings f ON c.Food_ID = f.Food_ID
        JOIN receivers r ON c.Receiver_ID = r.Receiver_ID
        GROUP BY r.Name
    """, conn)
    st.dataframe(df)
    show_chart(df)


elif query == "Q11: Most Claimed Meal Type":
    df = pd.read_sql("""
        SELECT f.Meal_Type, COUNT(*) AS claim_count
        FROM claims c
        JOIN food_listings f ON c.Food_ID = f.Food_ID
        GROUP BY f.Meal_Type
        ORDER BY claim_count DESC
    """, conn)
    st.dataframe(df)
    show_chart(df)


elif query == "Q12: City with Highest Claims":
    df = pd.read_sql("""
        SELECT f.Location, COUNT(*) AS total_claims
        FROM claims c
        JOIN food_listings f ON c.Food_ID = f.Food_ID
        GROUP BY f.Location
        ORDER BY total_claims DESC
        LIMIT 1
    """, conn)
    st.dataframe(df)


elif query == "Q13: Food Type Claimed the Most":
    df = pd.read_sql("""
        SELECT f.Food_Type, COUNT(*) AS total_claims
        FROM claims c
        JOIN food_listings f ON c.Food_ID = f.Food_ID
        GROUP BY f.Food_Type
        ORDER BY total_claims DESC
        LIMIT 1
    """, conn)
    st.dataframe(df)


elif query == "Q14: Average Quantity per Listing":
    df = pd.read_sql("""
        SELECT AVG(Quantity) AS avg_quantity
        FROM food_listings
    """, conn)
    st.dataframe(df)


elif query == "Q15: Provider-wise Average Donation":
    df = pd.read_sql("""
        SELECT p.Name, AVG(f.Quantity) AS avg_donation
        FROM food_listings f
        JOIN providers p ON f.Provider_ID = p.Provider_ID
        GROUP BY p.Name
    """, conn)
    st.dataframe(df)
    show_chart(df)



st.markdown("---")

st.caption("Food Redistribution System | Portfolio Project")


Overwriting FeedForward.py


In [ ]:
!streamlit run FeedForward.py